To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Read our **[Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

Okay, here's the Markdown for your first code snippet:

## 1. Environment Setup: Installing Dependencies

This cell handles the installation of essential Python libraries required for the project. It uses `pip` to install packages, with specific considerations for Google Colab environments.

*   **`unsloth`**: This is a key library for efficiently fine-tuning large language models (LLMs) like Mistral-7B, especially with limited resources. It provides optimizations for faster training and lower memory usage.
*   **`bitsandbytes`**: Used for 4-bit quantization, which significantly reduces the model's memory footprint, making it possible to fine-tune large models on consumer GPUs.
*   **`accelerate`**: A Hugging Face library that simplifies distributed training and running PyTorch code on various hardware configurations.
*   **`xformers`**: Provides memory-efficient attention mechanisms, further optimizing the training process.
*   **`peft` (Parameter-Efficient Fine-Tuning)**: Enables fine-tuning LLMs by training only a small subset of their parameters, which is much more efficient than full fine-tuning.
*   **`trl` (Transformer Reinforcement Learning)**: A Hugging Face library used here for its `SFTTrainer` (Supervised Fine-tuning Trainer), which simplifies the process of instruction fine-tuning.
*   **Other libraries**: `sentencepiece` (for tokenization), `protobuf`, `datasets` (for handling datasets), `huggingface_hub` (for interacting with the Hugging Face Hub), and `hf_transfer` (for faster model downloads) are also installed.

The conditional logic (`if "COLAB_" not in "".join(os.environ.keys())`) checks if the code is running in a Google Colab environment. Colab often requires specific versions or installation methods for certain packages (like `xformers` and `triton`) to ensure compatibility, hence the different `pip install` commands. The `--no-deps` flag is used in the Colab section to prevent conflicts with pre-installed Colab packages, installing only the specified package and not its dependencies.


In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
!pip install -qU wandb

In [ ]:
from google.colab import userdata


Okay, here's the Markdown for that code snippet:

## 2. Weights & Biases (W&B) Setup

This cell initializes the connection to **Weights & Biases (W&B)**. W&B is a popular platform for experiment tracking, dataset versioning, and model management in machine learning projects.

*   **`import wandb`**: This line imports the `wandb` library.
*   **`wandb.login(key=userdata.get('wandb'))`**: This line logs into your W&B account.
    *   It's crucial for tracking the fine-tuning process, allowing you to monitor metrics like loss, learning rate, and potentially evaluation scores in real-time through the W&B dashboard.
    *   `userdata.get('wandb')` is used to securely access your W&B API key. This is a good practice, especially in environments like Google Colab or Kaggle, where you can store secret keys without hardcoding them directly into the notebook. You would typically set this key in the "Secrets" manager of your notebook environment.

By logging into W&B, all subsequent training runs initiated with W&B integration will automatically send their logs, metrics, and model checkpoints (if configured) to your W&B project workspace. This helps in comparing different experiments, visualizing results, and collaborating with others.


In [ ]:
import wandb

In [ ]:
wandb.login(key=userdata.get('wandb'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amro-eidd (amro-eidd-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Unsloth

Okay, here's the Markdown for this code block:

## 3. Model and Tokenizer Initialization with Unsloth

This cell focuses on loading the pre-trained large language model (LLM) and its corresponding tokenizer. We're leveraging Unsloth's `FastLanguageModel` for optimized loading and memory efficiency, particularly when using 4-bit quantized models.

**Key Steps and Parameters:**

1.  **Import necessary libraries**:
    *   `FastLanguageModel` from `unsloth`: This is Unsloth's optimized class for loading and working with LLMs.
    *   `torch`: The PyTorch library, which is the underlying framework for many LLMs.

2.  **Configuration Parameters**:
    *   `max_seq_length = 2048`: This defines the maximum number of tokens the model can process in a single input sequence. Unsloth handles RoPE (Rotary Positional Embedding) scaling internally, allowing flexibility in this choice. For essay grading, a sufficiently large `max_seq_length` is important to accommodate the question, reference answer, student answer, and mark scheme.
    *   `dtype = None`: This setting allows Unsloth to automatically detect the optimal data type (e.g., `float16` for Tesla T4/V100 GPUs, `bfloat16` for Ampere+ GPUs) for the model based on the available hardware. This can improve performance and reduce memory usage.
    *   `load_in_4bit = True`: This crucial parameter instructs Unsloth to load the model using 4-bit quantization. Quantization reduces the model's precision (and thus its size and memory footprint) with minimal impact on performance for many tasks. This makes it feasible to run larger models on consumer-grade hardware.

3.  **`fourbit_models` List (Informational)**:
    *   This list showcases a variety of 4-bit quantized models available directly through Unsloth. These models are pre-quantized, leading to significantly faster download times and reduced risk of Out-Of-Memory (OOM) errors. The project specifies using a Mistral-7B variant.

4.  **Loading the Model and Tokenizer**:
    *   `model, tokenizer = FastLanguageModel.from_pretrained(...)`: This is the core command that loads the model and tokenizer.
        *   `model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"`: This specifies the exact model to be loaded from the Hugging Face Hub. We are using a 4-bit quantized version of Mistral-7B Instruct v0.3 provided by Unsloth, which is suitable for instruction-following tasks like essay grading.
        *   The `max_seq_length`, `dtype`, and `load_in_4bit` parameters defined earlier are passed here.
        *   The `token` argument (commented out) would be used if loading a "gated" model from Hugging Face that requires authentication (e.g., some Llama models).

**Output:**

*   `model`: The loaded 4-bit quantized Mistral-7B instruction-tuned model, ready for fine-tuning or inference.
*   `tokenizer`: The tokenizer associated with the Mistral model, responsible for converting text data into a format the model can understand (tokens) and vice-versa.

By using Unsloth and 4-bit quantization, we aim for faster training and reduced memory consumption, which is especially beneficial for this project.


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.8: Fast Mistral patching. Transformers: 4.52.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

Okay, here's the Markdown for configuring PEFT with LoRA:

## 4. Parameter-Efficient Fine-Tuning (PEFT) with LoRA

After loading the base model, we now apply Parameter-Efficient Fine-Tuning (PEFT) techniques, specifically LoRA (Low-Rank Adaptation), to prepare the model for our essay grading task. PEFT allows us to fine-tune the LLM efficiently by training only a small fraction of the model's parameters, rather than all of them. This significantly reduces computational cost and memory requirements while still achieving strong performance.

Unsloth's `FastLanguageModel.get_peft_model` method simplifies this process.

**Key Configuration Parameters for LoRA:**

*   **`model`**: The pre-trained model loaded in the previous step.
*   **`r = 32`**: This is the rank of the LoRA decomposition. A higher rank means more trainable parameters, potentially leading to better adaptation but also increasing computational cost. Common values are 8, 16, 32, 64, 128. `32` is a reasonable starting point.
*   **`target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`**: This list specifies which layers (or modules) of the transformer architecture will have LoRA adapters applied. These typically include the query, key, value, and output projections in the attention mechanisms, as well as parts of the feed-forward networks. Unsloth automatically identifies these common layers for Mistral-type models.
*   **`lora_alpha = 32`**: This is a scaling factor for the LoRA weights. It's often set to be the same as `r` or double `r`. It helps balance the influence of the pre-trained weights and the newly learned LoRA weights.
*   **`lora_dropout = 0`**: Dropout rate applied to the LoRA layers. While any value is supported, Unsloth optimizes for `0`. Dropout can help prevent overfitting.
*   **`bias = "none"`**: Specifies how biases are handled in the LoRA layers. `"none"` is an optimized setting in Unsloth, meaning biases are not trained for the LoRA adapters.
*   **`use_gradient_checkpointing = "unsloth"`**: This is a memory-saving technique. Instead of storing all intermediate activations during the forward pass (which consumes a lot of VRAM), gradient checkpointing recomputes them during the backward pass. The `"unsloth"` option enables Unsloth's highly optimized version of gradient checkpointing, which can lead to significant VRAM savings (e.g., ~30% less VRAM claimed by Unsloth), allowing for larger batch sizes or longer sequences, especially beneficial for tasks with long contexts like essay grading. Setting it to `True` would use PyTorch's native gradient checkpointing.
*   **`random_state = 3407`**: Sets a seed for reproducibility in the initialization of LoRA parameters.
*   **`use_rslora = False`**: Rank-Stabilized LoRA (RSLora) is an advanced technique that can improve stability and performance; it's disabled here.
*   **`loftq_config = None`**: LoftQ (LoRA Fine-Tuning with Quantization) is another advanced quantization-aware fine-tuning technique; it's not used in this configuration.

By applying these settings, we create a new version of the `model` where LoRA adapters are injected. During fine-tuning, only these adapters (and potentially other small components like the language model head, if we were doing classification) will be updated, making the training process much more efficient.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [ ]:
import numpy as np
import torch
import re
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from scipy.stats import pearsonr, spearmanr
import json

## 5. Preparing Data: Instruction Prompt Formatting

To effectively instruction-tune the model, we need to format our dataset entries into a specific prompt structure that the model expects. This structure typically includes an instruction, an input (providing context), and the desired output (the response we want the model to learn to generate).

This cell defines:
1.  An **instruction prompt template** (`alpaca_prompt`).
2.  A **function** (`formatting_prompts_func`) to apply this template to our dataset.

**1. The `alpaca_prompt` Template:**

```python
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

## 6. Evaluation: Helper Functions for Metrics Calculation

To assess the performance of our fine-tuned essay grading model, we need robust evaluation metrics. This section defines helper functions to:
1.  Extract numerical scores from the model's textual responses.
2.  Compute a comprehensive set of metrics comparing the model's predicted scores and rationales against the ground truth.
3.  Provide a fallback for dummy metrics in case of errors during computation.

These functions will be crucial for the `Trainer` during the evaluation phase.

### 6.1. `extract_score_from_response(response_text)`

This function is designed to parse the textual output generated by the language model and extract a numerical score. Since the model generates free-form text for the "Response" part (which includes both the score and the rationale), we need a reliable way to find the score.

**Functionality:**

1.  **Pattern Matching:** It uses a list of regular expression (`re`) patterns (`score_patterns`) to search for common ways a score might be presented in the text. Examples include:
    *   `"Score: 4"`
    *   `"score = 3"`
    *   `"score is 5"`
    *   `"4/5"` (extracting the numerator)
    *   `"Final Score: 10"`
2.  **Iterative Search:** It iterates through these patterns. If a match is found, it extracts the numerical part (the first capturing group `match.group(1)`) and converts it to an integer.
3.  **Fallback - General Number Search:** If none of the specific patterns match, it attempts a more general search for any standalone numbers (`\b\d+\b`) in the response.
4.  **Reasonable Score Check:** If numbers are found via the fallback method, it iterates through them and returns the first number that falls within a plausible score range (0-100). This helps filter out irrelevant numbers that might appear in the rationale.
5.  **Return Value:**
    *   Returns the extracted integer score if found.
    *   Returns `None` if no score can be reliably extracted.

This function is vital because the model's output for the "Response" (score and rationale) is learned during fine-tuning, and it might not always produce the score in a perfectly structured way. Robust extraction is key for accurate evaluation.

### 6.2. `compute_metrics(eval_preds)`

This is the core function for evaluating the model's performance on the essay grading task. It's designed to be used with the Hugging Face `Trainer` or `SFTTrainer`, which provides `eval_preds` (a tuple of predictions and labels) during the evaluation loop.

**Key Steps and Functionality:**

1.  **Input Handling (`eval_preds`):**
    *   Unpacks `predictions` and `labels` from `eval_preds`.
    *   Handles potential variations in the structure of `predictions` (e.g., if it's an object with a `.predictions` attribute).
    *   Converts predictions and labels to NumPy arrays if they are PyTorch tensors.

2.  **Decoding for SFTTrainer:**
    *   For Supervised Fine-Tuning (SFT) with `SFTTrainer`, the raw `predictions` are often logits (scores for each token in the vocabulary) across the sequence length. This step converts these logits to token IDs by taking the `argmax` along the vocabulary dimension (`np.argmax(predictions, axis=-1)`).
    *   Replaces special `-100` label values (used to ignore certain tokens during loss calculation, typically padding tokens) with the tokenizer's `pad_token_id`. This is necessary for correct decoding.

3.  **Token ID to Text Decoding:**
    *   Iterates through each pair of predicted token IDs and label token IDs.
    *   Filters out padding tokens from both prediction and label sequences.
    *   Uses `tokenizer.decode()` to convert the token ID sequences back into human-readable text (`decoded_preds` and `decoded_labels`). `skip_special_tokens=True` is used to remove tokens like `<s>`, `</s>`, etc., from the decoded text.
    *   Includes error handling (`try-except`) to skip examples that cause decoding issues.

4.  **Score Extraction:**
    *   If no valid predictions are decoded, it returns dummy metrics using `get_dummy_metrics()`.
    *   Iterates through the `decoded_preds` (model's generated text) and `decoded_labels` (ground truth text).
    *   Uses the `extract_score_from_response` function (defined above) to get the numerical scores from both the predicted text and the label text.
    *   For `true_score` extraction from labels, it includes an additional check for scores embedded in a JSON-like structure (e.g., `'"score": 4'`) within the label text, as this might be how the ground truth score is formatted in our dataset's "output" field.
    *   Keeps track of `successful_extractions` and `failed_extractions` of scores.

5.  **Metric Calculation (if scores are extracted):**
    *   If no scores can be extracted, it returns dummy metrics.
    *   Converts lists of `pred_scores` and `true_scores` to NumPy arrays.
    *   Calculates a wide range of metrics:
        *   **Dataset Info:** Total samples, successful/failed extractions, min/max true scores.
        *   **Basic Error Metrics:** Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE).
        *   **Score Differences:** Mean and standard deviation of absolute score differences, score bias (average difference, indicating over/under-grading).
        *   **Accuracy Metrics:**
            *   Exact Accuracy (percentage of predictions identical to true scores).
            *   Accuracy within a tolerance (e.g., `within_1_accuracy` means predicted score is ±1 of the true score).
        *   **Correlation Metrics:** Pearson and Spearman rank correlation coefficients (and their p-values) to measure the linear and monotonic relationship between predicted and true scores. Handles cases with insufficient variance.
        *   **Agreement Metrics:** Cohen's Kappa to measure inter-rater agreement (treating the model as one rater and human grades as another).
        *   **Classification-style Metrics:** Weighted F1-score (treating scores as classes).
        *   **Per-Score Performance (Simplified):** If the number of unique scores is small, it calculates precision, recall, F1-score, and support for each individual score value.
        *   **Grading Quality Assessment:** Counts and ratios of correctly graded, over-graded, and under-graded essays.
        *   **Mean Score Statistics:** Mean and standard deviation of true scores and predicted scores.

6.  **Error Handling:**
    *   A global `try-except` block wraps the entire function. If any unhandled error occurs, it prints an error message and returns dummy metrics.

This comprehensive set of metrics provides a multi-faceted view of the model's grading performance, going beyond simple accuracy to understand error magnitudes, correlation with human grades, and potential biases.

### 6.3. `get_dummy_metrics()`

This is a simple utility function that returns a dictionary of predefined "dummy" or default metric values (mostly zeros).

**Purpose:**

*   It is called by `compute_metrics` in two scenarios:
    1.  If there's a critical error during the metric computation process.
    2.  If no valid scores can be extracted from the model's predictions or the labels, making metric calculation impossible.
*   Ensures that the `Trainer` always receives a dictionary of metrics, even if meaningful evaluation couldn't be performed for a particular batch or run. This prevents the training/evaluation loop from crashing due to missing metric keys.

The selected dummy metrics cover some of the key metrics reported by `compute_metrics`, ensuring consistency in the keys expected by W&B or other logging mechanisms.

In [ ]:


def extract_score_from_response(response_text):
    """Extract numerical score from model response"""
    # Look for patterns like "Score: 4", "score: 3", "Score = 5", etc.
    score_patterns = [
        r"(?:score|Score):\s*(\d+)",
        r"(?:score|Score)\s*=\s*(\d+)",
        r"(?:score|Score)\s*is\s*(\d+)",
        r"(\d+)/\d+",  # For patterns like "4/5"
        r"(?:final|Final)?\s*(?:score|Score):\s*(\d+)"
    ]

    for pattern in score_patterns:
        match = re.search(pattern, response_text)
        if match:
            return int(match.group(1))

    # If no pattern matches, try to find any number in the response
    numbers = re.findall(r'\b\d+\b', response_text)
    if numbers:
        # Return the first reasonable score (assuming scores are 0-10 or 0-100)
        for num in numbers:
            score = int(num)
            if 0 <= score <= 100:  # Reasonable score range
                return score

    return None  # No score found

def compute_metrics(eval_preds):
    """
    Compute comprehensive evaluation metrics for essay grading
    For SFTTrainer, we need to handle the predictions differently
    """
    try:
        predictions, labels = eval_preds

        # Handle different prediction formats
        if hasattr(predictions, 'predictions'):
            predictions = predictions.predictions

        # Convert predictions to the right format
        if isinstance(predictions, torch.Tensor):
            predictions = predictions.cpu().numpy()
        if isinstance(labels, torch.Tensor):
            labels = labels.cpu().numpy()

        # For SFT training, predictions are logits, we need to get the argmax
        if len(predictions.shape) == 3:  # (batch_size, seq_len, vocab_size)
            predictions = np.argmax(predictions, axis=-1)

        # Replace -100 labels with pad token id for decoding
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)

        # Decode predictions and labels
        decoded_preds = []
        decoded_labels = []

        for pred_ids, label_ids in zip(predictions, labels):
            # Convert to lists and filter out padding tokens
            pred_ids = [int(id) for id in pred_ids if id != tokenizer.pad_token_id]
            label_ids = [int(id) for id in label_ids if id != tokenizer.pad_token_id]

            # Decode
            try:
                pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
                label_text = tokenizer.decode(label_ids, skip_special_tokens=True)
                decoded_preds.append(pred_text)
                decoded_labels.append(label_text)
            except Exception as e:
                # Skip problematic examples
                continue

        if not decoded_preds:
            # Return dummy metrics if no valid predictions
            return get_dummy_metrics()

        # Extract scores from predictions and ground truth
        pred_scores = []
        true_scores = []
        successful_extractions = 0
        failed_extractions = 0

        for pred_text, label_text in zip(decoded_preds, decoded_labels):
            # Extract predicted score
            pred_score = extract_score_from_response(pred_text)
            true_score = extract_score_from_response(label_text)

            if true_score is None:
                # Try to extract from JSON-like structure if present
                try:
                    score_match = re.search(r'"score":\s*(\d+)', label_text)
                    if score_match:
                        true_score = int(score_match.group(1))
                except:
                    pass

            if pred_score is not None and true_score is not None:
                pred_scores.append(pred_score)
                true_scores.append(true_score)
                successful_extractions += 1
            else:
                failed_extractions += 1

        if not pred_scores:
            return get_dummy_metrics()

        # Convert to numpy arrays
        pred_scores = np.array(pred_scores)
        true_scores = np.array(true_scores)

        # Calculate comprehensive metrics
        metrics = {}

        # Dataset Info
        total_samples = len(pred_scores)
        metrics['total_samples'] = float(total_samples)
        metrics['successful_extractions'] = float(successful_extractions)
        metrics['failed_extractions'] = float(failed_extractions)
        metrics['score_min'] = float(np.min(true_scores))
        metrics['score_max'] = float(np.max(true_scores))

        # Basic Error Metrics
        metrics['mae'] = float(mean_absolute_error(true_scores, pred_scores))
        metrics['mse'] = float(mean_squared_error(true_scores, pred_scores))
        metrics['rmse'] = float(np.sqrt(metrics['mse']))

        # Score differences
        score_diff = pred_scores - true_scores
        metrics['mean_score_difference'] = float(np.mean(np.abs(score_diff)))
        metrics['std_score_difference'] = float(np.std(score_diff))
        metrics['score_bias'] = float(np.mean(score_diff))  # positive = over-grading

        # Accuracy Metrics
        metrics['exact_accuracy'] = float(accuracy_score(true_scores, pred_scores))
        metrics['tolerance_0'] = metrics['exact_accuracy']  # Same as exact accuracy

        # Tolerance accuracies
        within_1 = np.abs(pred_scores - true_scores) <= 1
        within_2 = np.abs(pred_scores - true_scores) <= 2
        metrics['within_1_accuracy'] = float(np.mean(within_1))
        metrics['within_2_accuracy'] = float(np.mean(within_2))
        metrics['tolerance_1'] = metrics['within_1_accuracy']
        metrics['tolerance_2'] = metrics['within_2_accuracy']

        # Correlation Metrics
        if len(set(true_scores)) > 1 and len(set(pred_scores)) > 1:
            pearson_corr, pearson_p = pearsonr(true_scores, pred_scores)
            spearman_corr, spearman_p = spearmanr(true_scores, pred_scores)

            metrics['pearson_correlation'] = float(pearson_corr) if not np.isnan(pearson_corr) else 0.0
            metrics['pearson_p_value'] = float(pearson_p) if not np.isnan(pearson_p) else 1.0
            metrics['spearman_correlation'] = float(spearman_corr) if not np.isnan(spearman_corr) else 0.0
            metrics['spearman_p_value'] = float(spearman_p) if not np.isnan(spearman_p) else 1.0
        else:
            metrics['pearson_correlation'] = 0.0
            metrics['pearson_p_value'] = 1.0
            metrics['spearman_correlation'] = 0.0
            metrics['spearman_p_value'] = 1.0

        # Agreement Metrics
        try:
            metrics['cohen_kappa'] = float(cohen_kappa_score(true_scores, pred_scores))
        except:
            metrics['cohen_kappa'] = 0.0

        # Weighted F1 Score
        try:
            metrics['weighted_f1'] = float(f1_score(true_scores, pred_scores, average='weighted'))
        except:
            metrics['weighted_f1'] = 0.0

        # Per-score performance (simplified for metrics)
        unique_scores = sorted(list(set(true_scores.tolist() + pred_scores.tolist())))
        if len(unique_scores) <= 10:  # Only compute if reasonable number of classes
            try:
                precision, recall, f1, support = precision_recall_fscore_support(
                    true_scores, pred_scores, labels=unique_scores, average=None, zero_division=0
                )

                for i, score in enumerate(unique_scores):
                    if i < len(precision):  # Safety check
                        metrics[f'score_{score}_precision'] = float(precision[i])
                        metrics[f'score_{score}_recall'] = float(recall[i])
                        metrics[f'score_{score}_f1'] = float(f1[i])
                        metrics[f'score_{score}_support'] = float(support[i])
            except:
                pass  # Skip per-score metrics if they fail

        # Grading Quality Assessment
        correctly_graded = np.sum(pred_scores == true_scores)
        over_graded = np.sum(pred_scores > true_scores)
        under_graded = np.sum(pred_scores < true_scores)

        metrics['correctly_graded'] = float(correctly_graded)
        metrics['over_graded'] = float(over_graded)
        metrics['under_graded'] = float(under_graded)
        metrics['correctly_graded_ratio'] = float(correctly_graded / total_samples)
        metrics['over_graded_ratio'] = float(over_graded / total_samples)
        metrics['under_graded_ratio'] = float(under_graded / total_samples)

        # Mean score statistics (keeping original names for compatibility)
        metrics['mean_true_score'] = float(np.mean(true_scores))
        metrics['mean_pred_score'] = float(np.mean(pred_scores))
        metrics['std_true_score'] = float(np.std(true_scores))
        metrics['std_pred_score'] = float(np.std(pred_scores))

        return metrics

    except Exception as e:
        print(f"Error in compute_metrics: {e}")
        return get_dummy_metrics()

def get_dummy_metrics():
    """Return dummy metrics in case of error"""
    return {
        'mae': 0.0,
        'exact_accuracy': 0.0,
        'within_1_accuracy': 0.0,
        'pearson_correlation': 0.0,
        'total_samples': 0.0,
        'successful_extractions': 0.0,
        'failed_extractions': 0.0,
        'cohen_kappa': 0.0,
        'weighted_f1': 0.0,
        'correctly_graded_ratio': 0.0,
        'over_graded_ratio': 0.0,
        'under_graded_ratio': 0.0
    }



## 7. Dataset Loading, Preprocessing, and Splitting

This section details the process of loading our custom essay grading dataset, applying the previously defined prompt formatting, and splitting it into training, evaluation, and test sets. A well-structured dataset and appropriate splits are fundamental for successful model fine-tuning and reliable evaluation.

**Steps:**

1.  **Import `load_dataset`:**
    *   The `load_dataset` function from the `datasets` library (Hugging Face) is used to load data from various formats, including JSON.

2.  **Load the Dataset:**
    *   `dataset = load_dataset("json", data_files="Final_fixed2.json", split="train")`
        *   Loads data from a JSON file named `Final_fixed2.json`. It's assumed this file contains a list of JSON objects, where each object matches the specified `Dataset Format` (question, reference\_answer, student\_answer, mark\_scheme, score, rationale).
        *   `data_files="Final_fixed2.json"` specifies the source file.
        *   `split="train"` indicates that we are loading the entire content of this file as a single "train" split initially. If your JSON file itself had predefined splits, you could specify them here or load them separately.

3.  **Shuffle the Dataset:**
    *   `dataset = dataset.shuffle(seed=42)`
        *   Shuffles the entire dataset randomly. This is important to ensure that any inherent ordering in the original data file does not bias the training process or the splits.
        *   `seed=42` provides a fixed seed for the random shuffling, ensuring reproducibility. If you run this code again, the shuffle order will be the same.

4.  **Apply Prompt Formatting:**
    *   `dataset = dataset.map(formatting_prompts_func, batched=True)`
        *   Applies the `formatting_prompts_func` (defined in Section 5) to each example in the dataset.
        *   This function takes the raw `instruction`, `input`, and `output` fields from our dataset (which we would have prepared to align with the expected structure, e.g., by combining essay components into the 'input' field and score/rationale into the 'output' field before this step or by designing `formatting_prompts_func` to handle the original dataset fields directly) and formats them into the `alpaca_prompt` structure, creating a new field (typically named "text" by our function) containing the complete formatted prompt string ready for the model.
        *   `batched=True` processes examples in batches, which is generally more efficient.

5.  **Splitting the Dataset:**
    The dataset is split into three standard subsets:
    *   **Training set (`train_dataset`):** Used to fine-tune the model. (80% of the original data)
    *   **Evaluation set (`eval_dataset`):** Used during training to monitor performance on unseen data, tune hyperparameters, and decide when to stop training (early stopping). (10% of the original data)
    *   **Test set (`test_dataset`):** Held out completely until after training and used for a final, unbiased evaluation of the model's generalization ability. (10% of the original data)

    The splitting is done in two stages:
    *   **Stage 1:** `split_dataset = dataset.train_test_split(test_size=0.2, seed=42)`
        *   Splits the formatted dataset into an 80% training portion (`split_dataset["train"]`) and a 20% temporary portion (`split_dataset["test"]`).
    *   **Stage 2:** `temp_split = split_dataset["test"].train_test_split(test_size=0.5, seed=42)`
        *   Takes the 20% temporary portion and splits it equally (50/50) into the final evaluation set (`temp_split["train"]`) and the final test set (`temp_split["test"]`). This results in each of these sets being 10% of the original total dataset.

    The `seed=42` in `train_test_split` ensures that the splits are reproducible.

After these steps, `train_dataset`, `eval_dataset`, and `test_dataset` are ready to be used with the `SFTTrainer`.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="Final_fixed2.json", split="train")
dataset = dataset.shuffle(seed=42)
dataset = dataset.map(formatting_prompts_func, batched=True)

# Step 1: Split into 80% train and 20% temp (for eval + test)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
# Step 2: Split the 20% temp into 50% eval and 50% test (i.e., 10% each of the original full dataset)
temp_split = split_dataset["test"].train_test_split(test_size=0.5, seed=42)

# Final splits
train_dataset = split_dataset["train"]         # 80%
eval_dataset = temp_split["train"]             # 10%
test_dataset = temp_split["test"]              # 10%

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

## 8. Initializing the SFTTrainer for Fine-Tuning

With the model prepared (PEFT/LoRA applied) and the datasets loaded and formatted, we can now set up the `SFTTrainer` from the `trl` library. The `SFTTrainer` simplifies the process of Supervised Fine-Tuning for instruction-following language models.

**Key Components and Configuration:**

1.  **Import necessary classes:**
    *   `SFTTrainer` from `trl`: The main trainer class for supervised fine-tuning.
    *   `TrainingArguments` from `transformers`: A class to configure various aspects of the training process (batch size, learning rate, logging, saving, etc.).
    *   `is_bfloat16_supported` from `unsloth`: A utility to check if the current hardware supports `bfloat16` precision, which is beneficial for modern GPUs (Ampere architecture and newer).

2.  **Instantiate `SFTTrainer`:**
    The `SFTTrainer` is initialized with several key arguments:
    *   **`model = model`**: The PEFT-modified model we prepared earlier.
    *   **`tokenizer = tokenizer`**: The tokenizer corresponding to our base model.
    *   **`train_dataset = train_dataset`**: The formatted training dataset.
    *   **`eval_dataset = eval_dataset.select(range(40))`**: The formatted evaluation dataset.
        *   `eval_dataset.select(range(40))` is used here to select only the first 40 samples from the evaluation dataset. This is often done during development or initial runs to speed up the evaluation loop. For a full evaluation, you would typically use the entire `eval_dataset`.
    *   **`dataset_text_field = "text"`**: Specifies the name of the field in our datasets that contains the fully formatted prompt strings (which we created using `formatting_prompts_func` and named "text").
    *   **`max_seq_length = max_seq_length`**: The maximum sequence length the model will process, as defined earlier.
    *   **`dataset_num_proc = 2`**: The number of processes to use for dataset preprocessing (e.g., tokenization). Using multiple processes can speed up this step.
    *   **`packing = False`**: If set to `True`, `packing` is an optimization where multiple short sequences are concatenated and packed into a single longer sequence to better utilize the `max_seq_length`. This can significantly speed up training if your dataset has many short examples. For essay grading, where inputs are likely to be longer, `False` is a reasonable default, but this could be experimented with.
    *   **`compute_metrics = compute_metrics`**: This crucial argument passes our custom `compute_metrics` function (defined in Section 6.2) to the trainer. The trainer will call this function during evaluation to calculate and log the performance metrics.
    *   **`args = TrainingArguments(...)`**: This is where we configure the training process itself.

3.  **`TrainingArguments` Configuration:**
    This object controls various hyperparameters and settings for the training run:
    *   **Batching & Gradient Accumulation:**
        *   `per_device_train_batch_size = 2`: Number of training examples processed per GPU/device in one forward/backward pass.
        *   `per_device_eval_batch_size = 2`: Number of evaluation examples processed per GPU/device during evaluation.
        *   `gradient_accumulation_steps = 8`: Number of update steps to accumulate gradients over before performing a backward pass and updating model weights. The effective batch size becomes `per_device_train_batch_size * num_devices * gradient_accumulation_steps`. This allows training with larger effective batch sizes even with limited VRAM. Here, effective batch size for a single device would be `2 * 8 = 16`.
    *   **Training Duration & Warmup:**
        *   `warmup_steps = 20`: Number of steps at the beginning of training where the learning rate gradually increases from 0 to its initial value. This can help stabilize training.
        *   `max_steps = 200`: The total number of training steps to perform. This is an alternative to `num_train_epochs`. If set, it overrides `num_train_epochs`. (The commented out `num_train_epochs = 3` would train for 3 full passes over the training dataset if `max_steps` were not set). `200` steps is likely for a quick experimental run.
    *   **Learning Rate & Optimizer:**
        *   `learning_rate = 2e-4` (or 0.0002): The initial learning rate for the optimizer.
        *   `optim = "adamw_8bit"`: Specifies the AdamW optimizer, using its 8-bit variant for memory efficiency. Unsloth often works well with this.
        *   `weight_decay = 0.01`: Applies L2 regularization to prevent overfitting.
        *   `lr_scheduler_type = "cosine"`: The learning rate scheduler type. "cosine" annealing gradually reduces the learning rate following a cosine curve, which is often effective.
    *   **Precision (FP16/BF16):**
        *   `fp16 = not is_bfloat16_supported()`: Enables mixed-precision training with `float16` if `bfloat16` is *not* supported.
        *   `bf16 = is_bfloat16_supported()`: Enables mixed-precision training with `bfloat16` if it *is* supported (typically on Ampere GPUs and newer). Mixed precision can speed up training and reduce memory usage.
    *   **Logging & Saving:**
        *   `logging_steps = 1`: How often to log training metrics (e.g., loss). Logging every step can be verbose but useful for detailed monitoring.
        *   `output_dir = "outputs"`: Directory where model checkpoints and other outputs (like tokenizer files) will be saved.
        *   `report_to = "wandb"`: Instructs the trainer to log metrics and results to Weights & Biases. This requires `wandb.login()` to have been called earlier.
        *   `run_name = "llama-sft-experiment-v1"`: A custom name for this specific training run, which will appear in W&B. (Note: The model is Mistral, not Llama, so this might be a slight misnomer or placeholder).
        *   `eval_strategy = "steps"`: Perform evaluation at regular step intervals.
        *   `eval_steps = 10`: Evaluate the model every 10 training steps.
        *   `save_steps = 10`: Save a model checkpoint every 10 training steps. This can be frequent; for longer runs, this might be increased.
        *   `greater_is_better = False`: Indicates to the trainer whether the primary metric for comparing models (`eval_loss` by default, unless `metric_for_best_model` is set) is better when it's higher or lower. For loss, lower is better. If you were using accuracy, higher would be better.
    *   **Reproducibility:**
        *   `seed = 3407`: Sets the random seed for training to ensure reproducibility.

This `SFTTrainer` instance is now configured and ready to start the fine-tuning process using the `trainer.train()` method.

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset.select(range(40)),
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    compute_metrics = compute_metrics,  # Add the compute_metrics function
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        # num_train_epochs = 3, # Set this for 1 full training run.
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # Use this for WandB etc
        run_name = "llama-sft-experiment-v1",
        eval_strategy = "steps",
        eval_steps = 10,
        save_steps = 10,
        # load_best_model_at_end = True,
        logging_strategy="steps",
        # logging_steps=1,         # INCREASED from 1 - log every 10 steps
        # logging_first_step=True,  # Log the first step
        # logging_dir="./logs",     # Directory for logs
        greater_is_better = False,

    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/2841 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/40 [00:00<?, ? examples/s]

## 9. Initial GPU Memory Check

Before starting the computationally intensive fine-tuning process, it's good practice to check the current GPU memory status. This helps in understanding the baseline memory usage and the available capacity.

**Code Explanation:**

1.  **`gpu_stats = torch.cuda.get_device_properties(0)`**:
    *   Retrieves properties of the GPU with device ID 0 (assuming a single GPU setup or interest in the first GPU).
    *   `gpu_stats` will be an object containing information like the GPU name (`gpu_stats.name`) and total memory (`gpu_stats.total_memory`).

2.  **`start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)`**:
    *   `torch.cuda.max_memory_reserved()`: Returns the maximum GPU memory (in bytes) that has been reserved by the PyTorch allocator since the start of the session or the last reset. This includes memory allocated for models, tensors, and CUDA contexts.
    *   The value is converted from bytes to Gigabytes (GB) by dividing by `1024^3`.
    *   `round(..., 3)` rounds the result to three decimal places for readability.
    *   This `start_gpu_memory` essentially shows how much GPU VRAM is already in use by PyTorch *before* we begin the main training loop.

3.  **`max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)`**:
    *   `gpu_stats.total_memory`: Gets the total memory capacity (in bytes) of the GPU.
    *   This value is also converted to GB and rounded.

4.  **`print(...)` statements**:
    *   Display the GPU name, its total memory capacity, and the amount of memory currently reserved by PyTorch.

This output provides a snapshot of the GPU resources. During training, memory usage will increase as data batches are loaded, gradients are computed, and optimizer states are stored. Monitoring this helps in debugging Out-Of-Memory (OOM) errors and optimizing batch sizes or model configurations.

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.279 GB of memory reserved.


## 10. Weights & Biases Project Configuration

This cell sets up environment variables for Weights & Biases (W&B) to organize the experiment tracking.

*   `PROJECT_NAME`: Defines the overarching W&B project name where all runs related to this endeavor will be grouped.
*   `RUN_NAME`: Specifies a name for this particular training run.
*   `os.environ["WANDB_PROJECT"] = PROJECT_NAME`: Sets the W&B project name as an environment variable, which `wandb` can automatically pick up.

In [ ]:
PROJECT_NAME = "Mistral-instruction-tuning"
RUN_NAME = f"Mistral-7b-sft"

os.environ["WANDB_PROJECT"] = PROJECT_NAME

## 11. Model Fine-Tuning

This cell initiates the actual fine-tuning process of the model.

*   **`trainer_stats = trainer.train()`**:
    *   Calls the `train()` method of the `SFTTrainer` instance (`trainer`) we configured earlier.
    *   This starts the training loop, which will:
        *   Iterate through the `train_dataset` for the specified number of steps (`max_steps`) or epochs.
        *   Perform forward and backward passes.
        *   Update the model's LoRA weights based on the calculated loss.
        *   Periodically evaluate the model on the `eval_dataset` (as per `eval_strategy` and `eval_steps`).
        *   Log metrics (loss, custom metrics from `compute_metrics`, learning rate, etc.) to the console and to Weights & Biases (since `report_to="wandb"` was set).
        *   Save model checkpoints (as per `save_steps`).
    *   The training will continue until `max_steps` is reached.
    *   `trainer_stats` will store information and statistics about the completed training run, such as total training time, average training loss, etc.

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,841 | Num Epochs = 2 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080/7,000,000,000 (1.20% trained)


Step,Training Loss,Validation Loss,Mae,Mse,Rmse,Exact Accuracy,Within 1 Accuracy,Within 2 Accuracy,Pearson Correlation,Pearson P Value,Spearman Correlation,Spearman P Value,Mean True Score,Mean Pred Score,Std True Score,Std Pred Score
10,0.769700,0.781118,0.550000,0.650000,0.806226,0.500000,0.950000,1.000000,0.844585,0.000000,0.855143,0.000000,1.800000,1.350000,1.249000,1.038027
20,0.705400,0.666702,0.325000,0.325000,0.570088,0.675000,1.000000,1.000000,0.897220,0.000000,0.893129,0.000000,1.800000,1.825000,1.249000,1.262686


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


KeyboardInterrupt: 

In [ ]:
# test_results = evaluate_model_on_test_set(model, tokenizer, test_dataset)
# print("Test Results:", test_results)

## 12. Post-Training Resource Usage Statistics

This cell calculates and displays key statistics about the GPU memory usage and time taken during the fine-tuning process that just completed.

*   It captures the peak GPU memory reserved by PyTorch after training.
*   Calculates the memory specifically used during the LoRA training phase by subtracting the initial memory.
*   Prints the total training time in seconds and minutes (obtained from `trainer_stats`).
*   Reports peak memory usage in GB and as a percentage of total GPU memory, both for overall usage and for the training phase itself.

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



## 13. Inference: Using the Fine-Tuned Model for Essay Grading

Now that the model has been fine-tuned, this section demonstrates how to use it to grade a new, unseen essay. This involves preparing the input in the same instruction format used during training and then using the model to generate a score and rationale.

**Steps:**

1.  **Enable Inference Mode with Unsloth:**
    *   `FastLanguageModel.for_inference(model)`: This Unsloth utility optimizes the fine-tuned model specifically for faster inference. It might involve operations like merging LoRA adapters back into the base model weights (if applicable and desired for deployment) or other optimizations to speed up generation.

2.  **Prepare Input for the Model:**
    *   The `alpaca_prompt` template (defined in Section 5) is used again.
    *   **Instruction:** A clear instruction is provided: `"Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale."`
    *   **Input:** This section contains the context for the grading task:
        *   `Question`: The essay question.
        *   `Reference Answer`: An expert/ideal answer.
        *   `Student Answer`: The student's essay to be graded.
        *   `Mark Scheme`: The criteria for grading.
        *   All these components are formatted into a single string.
    *   **Output/Response:** The `{}` placeholder for the response in the `alpaca_prompt` is left empty (`""`). This is crucial because we want the model to *generate* this part.
    *   **Tokenization:**
        *   `inputs = tokenizer([...], return_tensors="pt").to("cuda")`:
            *   The formatted prompt string is passed to the `tokenizer`.
            *   `return_tensors="pt"` ensures the output is PyTorch tensors.
            *   `.to("cuda")` moves the tokenized input tensors to the GPU for faster processing.

3.  **Generate the Response (Score and Rationale):**
    *   `outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)`:
        *   The `model.generate()` method is called to produce the output.
        *   `**inputs`: Unpacks the tokenized input tensors (`input_ids`, `attention_mask`, etc.).
        *   `max_new_tokens=64`: Limits the maximum number of new tokens the model can generate for its response. This should be set appropriately to allow for both a score and a concise rationale.
        *   `use_cache=True`: Enables the use of a key/value cache during generation, which speeds up the decoding process for autoregressive models like Mistral.

4.  **Decode the Output:**
    *   `tokenizer.batch_decode(outputs)`:
        *   Converts the generated token IDs (`outputs`) back into human-readable text.
        *   The result will be a list containing the model's generated response, which should include the predicted score and the rationale.

This demonstrates a single inference example. In a real application, you would loop through your unseen essays, format them similarly, and use the model to get grades and explanations.

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
            "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.",  # instruction
        """Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the importance of different words.
          2. Mentions the ability to focus on relevant parts of input.
          3. Explains how attention captures context or relationships.
          4. Refers to handling long-range dependencies or position-independence.""",  # input
        "",  # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

["<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGrade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.\n\n### Input:\nQuestion: What is the role of attention mechanisms in transformer models?\n\n          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.\n\n          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.\n\n          Mark Scheme:\n          1. Describes attention as wei

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.",  # instruction
        """Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the importance of different words.
          2. Mentions the ability to focus on relevant parts of input.
          3. Explains how attention captures context or relationships.
          4. Refers to handling long-range dependencies or position-independence.""",  # input
        "",  # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the impor

<a name="Save"></a>
## Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.",  # instruction
        """Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the importance of different words.
          2. Mentions the ability to focus on relevant parts of input.
          3. Explains how attention captures context or relationships.
          4. Refers to handling long-range dependencies or position-independence.""",  # input
        "",  # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

==((====))==  Unsloth 2025.5.8: Fast Mistral patching. Transformers: 4.52.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load lora_model as a legacy tokenizer.


<s>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the import

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
import re
import torch
from transformers import TextStreamer
from datasets import load_dataset
import numpy as np
from tqdm import tqdm

In [ ]:

import wandb
import torch
import re
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, precision_recall_fscore_support, cohen_kappa_score
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

### 14.4. `evaluate_model(...)` - Comprehensive Evaluation Pipeline

This is the main function for conducting a thorough evaluation of the fine-tuned essay grading model. It integrates prediction generation, score extraction, metric calculation, and extensive logging to Weights & Biases.

**Key Responsibilities & Steps:**

1.  **W&B Initialization:**
    *   Calls `wandb.init()` to start a new W&B run for this evaluation.
    *   Configures the run with `project_name`, `run_name`, `model_name`, number of samples, and other relevant hyperparameters.

2.  **Data Sampling:**
    *   Determines the number of samples to evaluate from `eval_dataset` (either all or a specified `num_samples`).

3.  **Iterative Evaluation:**
    *   Loops through the selected samples from the `eval_dataset` (using `tqdm` for a progress bar).
    *   For each sample:
        *   Calls `generate_prediction()` to get the model's textual output.
        *   Uses `extract_score_from_output()` to get the `predicted_score`.
        *   Uses `extract_ground_truth_score()` to get the `ground_truth_score` from the example's 'output' field.
        *   If both scores are successfully extracted:
            *   Calculates if the prediction is correct and the absolute score difference.
            *   Stores scores for later aggregate metric calculation.
            *   Adds detailed information for the sample (ID, instruction, input, ground truth text, prediction text, scores, correctness, difference) to a `wandb.Table` (`examples_table`) for qualitative analysis (limited to the first 100 examples).
            *   Logs running accuracy and MAE to W&B every 50 examples.
        *   Handles and counts `failed_extractions` if scores cannot be parsed.
        *   Includes error handling for issues during prediction generation for a sample.

4.  **Metric Calculation (if successful predictions exist):**
    *   Calculates a comprehensive suite of metrics using `sklearn.metrics` and `numpy`:
        *   **Accuracy:** Exact match, tolerance-based (e.g., ±1 point, ±2 points).
        *   **Error Metrics:** MAE, MSE, RMSE, mean/std of score differences, score bias (average of `pred - true`).
        *   **Correlation:** Pearson and Spearman correlation coefficients.
        *   **Agreement:** Cohen's Kappa, Weighted F1-score.
        *   **Grading Quality:** Ratios of correctly graded, over-graded, and under-graded essays.
        *   **Dataset/Extraction Info:** Total samples, successful/failed extractions, success rate, score range.
        *   **Per-Score Metrics:** Precision, recall, F1, support for each unique score value.
        *   **Score Distribution Counts:** Frequency of each score in ground truth vs. predictions.

5.  **W&B Logging:**
    *   Logs all calculated scalar metrics to W&B under structured categories (e.g., "accuracy/", "error/", "correlation/").
    *   Generates and logs visualizations:
        *   Confusion matrix (`confusion_plot`)
        *   Score distribution comparison (`distribution_plot`)
        *   Scatter plot of predicted vs. true scores (`scatter_plot`)
        *   Histogram of score differences.
    *   Logs the `examples_table` containing detailed sample-wise results.
    *   Logs a summary table (`evaluation_summary`) of key metrics.

6.  **Console Output:**
    *   Prints a detailed summary of the evaluation results to the console.

7.  **Cleanup & Return:**
    *   Calls `wandb.finish()` to close the W&B run.
    *   Returns a dictionary containing key evaluation metrics.
    *   If no successful predictions are made, it logs an error to W&B and returns `None`.

This function provides a holistic view of the model's performance, essential for iterating on model development and understanding its capabilities in the essay grading context.

In [ ]:
def extract_score_from_output(text):
    """
    Extract score from model output using regex
    Looks for patterns like "Score: 4" or "Score:4"
    """
    # Pattern to match "Score: X" where X is a number
    score_pattern = r'[Ss]core:\s*(\d+)'
    match = re.search(score_pattern, text)

    if match:
        return int(match.group(1))
    else:
        # Try alternative patterns
        alt_patterns = [
            r'[Ss]core\s*=\s*(\d+)',  # Score = 4
            r'[Ss]core\s+(\d+)',      # Score 4
            r'(\d+)/\d+',             # 4/4 format
            r'[Ss]core:\s*(\d+\.?\d*)', # Score: 3.5
        ]

        for pattern in alt_patterns:
            match = re.search(pattern, text)
            if match:
                return int(float(match.group(1)))

        # If no score found, return None
        return None

def extract_ground_truth_score(output_text):
    """
    Extract score from ground truth output
    """
    return extract_score_from_output(output_text)

def generate_prediction(model, tokenizer, instruction, input_text):
    """
    Generate prediction for a single example
    """
    # Format the prompt (leave output blank for generation)
    formatted_prompt = alpaca_prompt.format(instruction, input_text, "")

    # Tokenize
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,  # Use greedy decoding for consistency
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response part (after "### Response:")
    response_start = generated_text.find("### Response:")
    if response_start != -1:
        response = generated_text[response_start + len("### Response:"):].strip()
    else:
        response = generated_text

    return response

def create_confusion_matrix_plot(ground_truth_scores, predicted_scores):
    """Create and return confusion matrix plot"""
    from sklearn.metrics import confusion_matrix

    unique_scores = sorted(set(ground_truth_scores + predicted_scores))
    cm = confusion_matrix(ground_truth_scores, predicted_scores, labels=unique_scores)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=unique_scores, yticklabels=unique_scores)
    plt.title('Confusion Matrix: Predicted vs Ground Truth Scores')
    plt.xlabel('Predicted Score')
    plt.ylabel('Ground Truth Score')
    plt.tight_layout()

    return plt

def create_score_distribution_plot(ground_truth_scores, predicted_scores):
    """Create score distribution comparison plot"""
    unique_scores = sorted(set(ground_truth_scores + predicted_scores))

    gt_counts = [ground_truth_scores.count(score) for score in unique_scores]
    pred_counts = [predicted_scores.count(score) for score in unique_scores]

    x = np.arange(len(unique_scores))
    width = 0.35

    plt.figure(figsize=(12, 6))
    plt.bar(x - width/2, gt_counts, width, label='Ground Truth', alpha=0.8)
    plt.bar(x + width/2, pred_counts, width, label='Predicted', alpha=0.8)

    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.title('Score Distribution: Ground Truth vs Predicted')
    plt.xticks(x, unique_scores)
    plt.legend()
    plt.tight_layout()

    return plt

def create_scatter_plot(ground_truth_scores, predicted_scores):
    """Create scatter plot of predictions vs ground truth"""
    plt.figure(figsize=(10, 8))
    plt.scatter(ground_truth_scores, predicted_scores, alpha=0.6)

    # Add perfect prediction line
    min_score = min(min(ground_truth_scores), min(predicted_scores))
    max_score = max(max(ground_truth_scores), max(predicted_scores))
    plt.plot([min_score, max_score], [min_score, max_score], 'r--', label='Perfect Prediction')

    plt.xlabel('Ground Truth Score')
    plt.ylabel('Predicted Score')
    plt.title('Predicted vs Ground Truth Scores')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    return plt

def evaluate_model(model, tokenizer, eval_dataset, num_samples=None,
                  project_name="educational-assessment", run_name=None,
                  model_name="unknown", config=None):
    """
    Evaluate the model on the eval dataset with educational assessment metrics
    Includes comprehensive wandb logging
    """
    # Initialize wandb
    wandb.init(
        project=project_name,
        name=run_name,
        config={
            "model_name": model_name,
            "num_samples": num_samples if num_samples else len(eval_dataset),
            "max_new_tokens": 128,
            "do_sample": False,
            **(config or {})
        }
    )

    if num_samples is None:
        num_samples = len(eval_dataset)
    else:
        num_samples = min(num_samples, len(eval_dataset))

    correct_predictions = 0
    total_predictions = 0
    score_differences = []
    failed_extractions = 0

    # For classification metrics
    predicted_scores = []
    ground_truth_scores = []

    results = []

    print(f"Evaluating on {num_samples} samples...")

    # Log examples table
    examples_table = wandb.Table(columns=[
        "example_id", "instruction", "input", "ground_truth", "prediction",
        "gt_score", "pred_score", "is_correct", "score_diff"
    ])

    for i in tqdm(range(num_samples)):
        example = eval_dataset[i]

        # Generate prediction
        try:
            prediction = generate_prediction(
                model, tokenizer,
                example['instruction'],
                example['input']
            )

            # Extract predicted score
            predicted_score = extract_score_from_output(prediction)

            # Extract ground truth score
            ground_truth_score = extract_ground_truth_score(example['output'])

            if predicted_score is not None and ground_truth_score is not None:
                # Check if prediction is correct
                is_correct = predicted_score == ground_truth_score
                correct_predictions += int(is_correct)
                total_predictions += 1

                # Calculate score difference
                score_diff = abs(predicted_score - ground_truth_score)
                score_differences.append(score_diff)

                # Store for metrics calculation
                predicted_scores.append(predicted_score)
                ground_truth_scores.append(ground_truth_score)

                # Store result
                result = {
                    'example_id': i,
                    'predicted_score': predicted_score,
                    'ground_truth_score': ground_truth_score,
                    'is_correct': is_correct,
                    'score_difference': score_diff,
                    'prediction': prediction[:200] + "..." if len(prediction) > 200 else prediction
                }
                results.append(result)

                # Add to wandb table (limit to first 100 examples to avoid large tables)
                if i < 100:
                    examples_table.add_data(
                        i,
                        example['instruction'][:100] + "..." if len(example['instruction']) > 100 else example['instruction'],
                        example['input'][:100] + "..." if len(example['input']) > 100 else example['input'],
                        example['output'][:100] + "..." if len(example['output']) > 100 else example['output'],
                        prediction[:100] + "..." if len(prediction) > 100 else prediction,
                        ground_truth_score,
                        predicted_score,
                        is_correct,
                        score_diff
                    )

                # Log running metrics every 50 examples
                if (i + 1) % 50 == 0:
                    running_accuracy = correct_predictions / total_predictions
                    running_mae = np.mean(score_differences)
                    wandb.log({
                        "running_accuracy": running_accuracy,
                        "running_mae": running_mae,
                        "examples_processed": total_predictions
                    })

            else:
                failed_extractions += 1
                print(f"Failed to extract score for example {i}")
                print(f"Prediction: {prediction[:100]}...")
                print(f"Ground truth: {example['output'][:100]}...")
                print("-" * 50)

        except Exception as e:
            print(f"Error processing example {i}: {str(e)}")
            failed_extractions += 1

    # Calculate comprehensive metrics
    if total_predictions > 0:
        # Basic metrics
        accuracy = correct_predictions / total_predictions
        mae = mean_absolute_error(ground_truth_scores, predicted_scores)
        mse = mean_squared_error(ground_truth_scores, predicted_scores)
        rmse = np.sqrt(mse)

        # Correlation metrics
        pearson_corr, pearson_p = pearsonr(ground_truth_scores, predicted_scores)
        spearman_corr, spearman_p = spearmanr(ground_truth_scores, predicted_scores)

        # Cohen's Kappa (agreement metric)
        kappa = cohen_kappa_score(ground_truth_scores, predicted_scores)

        # Weighted F1 score (treats as multi-class classification)
        weighted_f1 = f1_score(ground_truth_scores, predicted_scores, average='weighted')

        # Precision and Recall per class
        precision, recall, f1_per_class, support = precision_recall_fscore_support(
            ground_truth_scores, predicted_scores, average=None, zero_division=0
        )

        # Educational assessment specific metrics
        mean_score_diff = np.mean(score_differences)
        std_score_diff = np.std(score_differences)

        # Tolerance-based accuracy (important for grading systems)
        tolerance_0 = sum(1 for diff in score_differences if diff == 0) / total_predictions
        tolerance_1 = sum(1 for diff in score_differences if diff <= 1) / total_predictions
        tolerance_2 = sum(1 for diff in score_differences if diff <= 2) / total_predictions

        # Bias analysis
        score_bias = np.mean(np.array(predicted_scores) - np.array(ground_truth_scores))

        # Score distribution analysis
        unique_scores = sorted(set(ground_truth_scores + predicted_scores))

        # Grading quality assessment
        over_graded = sum(1 for p, g in zip(predicted_scores, ground_truth_scores) if p > g)
        under_graded = sum(1 for p, g in zip(predicted_scores, ground_truth_scores) if p < g)
        correctly_graded = sum(1 for p, g in zip(predicted_scores, ground_truth_scores) if p == g)

        # Create visualizations
        confusion_plot = create_confusion_matrix_plot(ground_truth_scores, predicted_scores)
        distribution_plot = create_score_distribution_plot(ground_truth_scores, predicted_scores)
        scatter_plot = create_scatter_plot(ground_truth_scores, predicted_scores)

        # Log main metrics to wandb
        metrics = {
            # Accuracy metrics
            "accuracy/exact_match": accuracy,
            "accuracy/tolerance_0": tolerance_0,
            "accuracy/tolerance_1": tolerance_1,
            "accuracy/tolerance_2": tolerance_2,

            # Error metrics
            "error/mae": mae,
            "error/mse": mse,
            "error/rmse": rmse,
            "error/mean_score_diff": mean_score_diff,
            "error/std_score_diff": std_score_diff,
            "error/score_bias": score_bias,

            # Correlation metrics
            "correlation/pearson": pearson_corr,
            "correlation/pearson_p_value": pearson_p,
            "correlation/spearman": spearman_corr,
            "correlation/spearman_p_value": spearman_p,

            # Agreement metrics
            "agreement/cohen_kappa": kappa,
            "agreement/weighted_f1": weighted_f1,

            # Grading analysis
            "grading/correctly_graded_ratio": correctly_graded / total_predictions,
            "grading/over_graded_ratio": over_graded / total_predictions,
            "grading/under_graded_ratio": under_graded / total_predictions,

            # Dataset info
            "dataset/total_samples": num_samples,
            "dataset/successful_extractions": total_predictions,
            "dataset/failed_extractions": failed_extractions,
            "dataset/success_rate": total_predictions / num_samples,
            "dataset/score_range_min": min(unique_scores),
            "dataset/score_range_max": max(unique_scores),
        }

        # Log per-score metrics
        for i, score in enumerate(unique_scores):
            if i < len(precision):
                metrics[f"per_score/score_{score}_precision"] = precision[i]
                metrics[f"per_score/score_{score}_recall"] = recall[i]
                metrics[f"per_score/score_{score}_f1"] = f1_per_class[i]
                metrics[f"per_score/score_{score}_support"] = support[i]

        # Log score distribution
        for score in unique_scores:
            gt_count = ground_truth_scores.count(score)
            pred_count = predicted_scores.count(score)
            metrics[f"distribution/gt_score_{score}"] = gt_count
            metrics[f"distribution/pred_score_{score}"] = pred_count

        wandb.log(metrics)

        # Log visualizations
        wandb.log({
            "confusion_matrix": wandb.Image(confusion_plot),
            "score_distribution": wandb.Image(distribution_plot),
            "predictions_scatter": wandb.Image(scatter_plot)
        })

        # Log examples table
        wandb.log({"evaluation_examples": examples_table})

        # Log score differences histogram
        plt.figure(figsize=(10, 6))
        plt.hist(score_differences, bins=max(1, max(score_differences) + 1), alpha=0.7)
        plt.xlabel('Score Difference (|predicted - ground_truth|)')
        plt.ylabel('Frequency')
        plt.title('Distribution of Score Differences')
        plt.grid(True, alpha=0.3)
        wandb.log({"score_differences_histogram": wandb.Image(plt)})

        # Close all plots
        plt.close('all')

        # Print comprehensive results
        print(f"\n=== COMPREHENSIVE EVALUATION RESULTS ===")
        print(f"Dataset Info:")
        print(f"  Total samples processed: {num_samples}")
        print(f"  Successful extractions: {total_predictions}")
        print(f"  Failed extractions: {failed_extractions}")
        print(f"  Score range: {min(unique_scores)} to {max(unique_scores)}")

        print(f"\nAccuracy Metrics:")
        print(f"  Exact Match Accuracy: {accuracy:.4f} ({correct_predictions}/{total_predictions})")
        print(f"  Tolerance-0 (Perfect): {tolerance_0:.4f}")
        print(f"  Tolerance-1 (±1 point): {tolerance_1:.4f}")
        print(f"  Tolerance-2 (±2 points): {tolerance_2:.4f}")

        print(f"\nError Metrics:")
        print(f"  Mean Absolute Error (MAE): {mae:.4f}")
        print(f"  Root Mean Square Error (RMSE): {rmse:.4f}")
        print(f"  Mean Score Difference: {mean_score_diff:.4f}")
        print(f"  Std Score Difference: {std_score_diff:.4f}")
        print(f"  Score Bias (pred - true): {score_bias:.4f}")

        print(f"\nCorrelation Metrics:")
        print(f"  Pearson Correlation: {pearson_corr:.4f} (p={pearson_p:.4f})")
        print(f"  Spearman Correlation: {spearman_corr:.4f} (p={spearman_p:.4f})")

        print(f"\nAgreement Metrics:")
        print(f"  Cohen's Kappa: {kappa:.4f}")
        print(f"  Weighted F1-Score: {weighted_f1:.4f}")

        # Per-score analysis
        print(f"\nPer-Score Performance:")
        for score in unique_scores:
            if score in ground_truth_scores:
                idx = unique_scores.index(score) if score < len(precision) else -1
                if idx >= 0 and idx < len(precision):
                    count = ground_truth_scores.count(score)
                    print(f"  Score {score}: Precision={precision[idx]:.3f}, Recall={recall[idx]:.3f}, F1={f1_per_class[idx]:.3f}, Support={support[idx]}")

        print(f"\nGrading Quality Assessment:")
        print(f"  Correctly graded: {correctly_graded}/{total_predictions} ({correctly_graded/total_predictions:.4f})")
        print(f"  Over-graded: {over_graded}/{total_predictions} ({over_graded/total_predictions:.4f})")
        print(f"  Under-graded: {under_graded}/{total_predictions} ({under_graded/total_predictions:.4f})")

        # Create summary table for wandb
        summary_table = wandb.Table(columns=["Metric", "Value"])
        summary_metrics = [
            ("Exact Match Accuracy", f"{accuracy:.4f}"),
            ("MAE", f"{mae:.4f}"),
            ("RMSE", f"{rmse:.4f}"),
            ("Pearson Correlation", f"{pearson_corr:.4f}"),
            ("Cohen's Kappa", f"{kappa:.4f}"),
            ("Tolerance ±1", f"{tolerance_1:.4f}"),
            ("Score Bias", f"{score_bias:.4f}")
        ]

        for metric, value in summary_metrics:
            summary_table.add_data(metric, value)

        wandb.log({"evaluation_summary": summary_table})

    else:
        print("No successful predictions to evaluate!")
        wandb.log({"error": "No successful predictions"})
        wandb.finish()
        return None

    # Finish wandb run
    wandb.finish()

    return {
        'accuracy': accuracy,
        'mae': mae,
        'rmse': rmse,
        'pearson_correlation': pearson_corr,
        'spearman_correlation': spearman_corr,
        'cohen_kappa': kappa,
        'weighted_f1': weighted_f1,
        'tolerance_1': tolerance_1,
        'tolerance_2': tolerance_2,
        'score_bias': score_bias,
        'total_predictions': total_predictions,
        'failed_extractions': failed_extractions,
        'results': results
    }

In [ ]:
test_dataset.shape

(356, 4)

## 15. Executing Evaluation and Interpreting Results

This final cell executes the comprehensive evaluation on the `test_dataset` using the `evaluate_model` function. After the evaluation completes, it prints a summary of the results, including example predictions and an interpretation guide for key metrics relevant to educational assessment.

**Execution:**

*   `evaluation_results = evaluate_model(model, tokenizer, test_dataset, num_samples=356)`:
    *   Calls the `evaluate_model` function (defined in Section 14.4).
    *   Passes the fine-tuned `model`, `tokenizer`, and the `test_dataset`.
    *   `num_samples=356` specifies that the evaluation should run on 356 samples from the test set. This could be the full test set or a subset for quicker evaluation. The `evaluate_model` function will handle the W&B initialization for this specific evaluation run.

**Output and Interpretation (if `evaluation_results` is not `None`):**

1.  **Example Predictions:**
    *   Prints the predicted score, ground truth score, correctness, and the model's full textual prediction (rationale + score) for the first 5 examples from the evaluation run. This allows for a qualitative check of the model's behavior.

2.  **Key Metrics Summary:**
    *   Displays a concise summary of metrics particularly relevant for automated essay grading:
        *   **Exact Match Accuracy:** Percentage of predictions where the model's score exactly matches the human score.
        *   **Within ±1 Point Accuracy (`tolerance_1`):** Percentage of predictions where the model's score is within one point (above or below) of the human score. This is often a more practical measure of agreement in grading.
        *   **Mean Absolute Error (MAE):** The average absolute difference between predicted and human scores. Lower is better.
        *   **Score Correlation (Pearson):** Measures the linear relationship between predicted and human scores. Higher (closer to 1) is better.
        *   **Inter-rater Agreement (Cohen's κ):** Measures the agreement between the model and human graders, correcting for chance agreement. Higher is better.

3.  **Interpretation Guide:**
    *   Provides a qualitative interpretation for Cohen's Kappa, MAE, and ±1 Point Accuracy based on common heuristic thresholds. This helps to contextualize the numerical results:
        *   **Cohen's Kappa:** Categorizes the level of agreement (e.g., "Substantial agreement," "Moderate agreement").
        *   **MAE:** Assesses the grading precision (e.g., "Excellent grading precision," "Acceptable grading precision").
        *   **±1 Point Accuracy:** Evaluates the practical accuracy for real-world use (e.g., "Excellent practical accuracy," "Good practical accuracy").

This final step provides both quantitative metrics and qualitative insights into the performance of the automatic essay grading system on unseen data, concluding the evaluation phase of the project. The results logged to Weights & Biases during the `evaluate_model` call offer a more in-depth and persistent record of this evaluation.

In [ ]:
evaluation_results = evaluate_model(model, tokenizer, test_dataset, num_samples=356)  # Start with 50 samples

if evaluation_results:
    # Print some example predictions
    print("\n=== EXAMPLE PREDICTIONS ===")
    for i, result in enumerate(evaluation_results['results'][:5]):  # Show first 5 examples
        print(f"\nExample {i+1}:")
        print(f"Predicted Score: {result['predicted_score']}")
        print(f"Ground Truth Score: {result['ground_truth_score']}")
        print(f"Correct: {result['is_correct']}")
        print(f"Prediction: {result['prediction']}")
        print("-" * 80)

    # Summary of key metrics for educational assessment
    print(f"\n=== KEY METRICS SUMMARY ===")
    print(f"Model Performance for Automated Grading:")
    print(f"  • Exact Match Accuracy: {evaluation_results['accuracy']:.3f}")
    print(f"  • Within ±1 Point Accuracy: {evaluation_results['tolerance_1']:.3f}")
    print(f"  • Mean Absolute Error: {evaluation_results['mae']:.3f} points")
    print(f"  • Score Correlation (Pearson): {evaluation_results['pearson_correlation']:.3f}")
    print(f"  • Inter-rater Agreement (Cohen's κ): {evaluation_results['cohen_kappa']:.3f}")

    # Interpretation guide
    print(f"\n=== INTERPRETATION GUIDE ===")
    kappa_val = evaluation_results['cohen_kappa']
    if kappa_val > 0.8:
        kappa_interp = "Almost perfect agreement"
    elif kappa_val > 0.6:
        kappa_interp = "Substantial agreement"
    elif kappa_val > 0.4:
        kappa_interp = "Moderate agreement"
    elif kappa_val > 0.2:
        kappa_interp = "Fair agreement"
    else:
        kappa_interp = "Poor agreement"

    print(f"Cohen's Kappa ({kappa_val:.3f}): {kappa_interp}")

    mae_val = evaluation_results['mae']
    if mae_val < 0.5:
        mae_interp = "Excellent grading precision"
    elif mae_val < 1.0:
        mae_interp = "Good grading precision"
    elif mae_val < 1.5:
        mae_interp = "Acceptable grading precision"
    else:
        mae_interp = "Needs improvement in grading precision"

    print(f"MAE ({mae_val:.3f}): {mae_interp}")

    tolerance_1 = evaluation_results['tolerance_1']
    if tolerance_1 > 0.9:
        tolerance_interp = "Excellent practical accuracy"
    elif tolerance_1 > 0.8:
        tolerance_interp = "Very good practical accuracy"
    elif tolerance_1 > 0.7:
        tolerance_interp = "Good practical accuracy"
    else:
        tolerance_interp = "Needs improvement for practical use"

    print(f"±1 Point Accuracy ({tolerance_1:.3f}): {tolerance_interp}")

Evaluating on 356 samples...


100%|██████████| 356/356 [18:25<00:00,  3.10s/it]



=== COMPREHENSIVE EVALUATION RESULTS ===
Dataset Info:
  Total samples processed: 356
  Successful extractions: 356
  Failed extractions: 0
  Score range: 0 to 5

Accuracy Metrics:
  Exact Match Accuracy: 0.8258 (294/356)
  Tolerance-0 (Perfect): 0.8258
  Tolerance-1 (±1 point): 0.9972
  Tolerance-2 (±2 points): 0.9972

Error Metrics:
  Mean Absolute Error (MAE): 0.1798
  Root Mean Square Error (RMSE): 0.4434
  Mean Score Difference: 0.1798
  Std Score Difference: 0.4054
  Score Bias (pred - true): 0.0225

Correlation Metrics:
  Pearson Correlation: 0.9399 (p=0.0000)
  Spearman Correlation: 0.9333 (p=0.0000)

Agreement Metrics:
  Cohen's Kappa: 0.7660
  Weighted F1-Score: 0.8241

Per-Score Performance:
  Score 0: Precision=0.839, Recall=0.810, F1=0.825, Support=58
  Score 1: Precision=0.788, Recall=0.745, F1=0.766, Support=55
  Score 2: Precision=0.817, Recall=0.790, F1=0.803, Support=62
  Score 3: Precision=0.836, Recall=0.914, F1=0.873, Support=139
  Score 4: Precision=0.829, Recall

accuracy/exact_match,▁
accuracy/tolerance_0,▁
accuracy/tolerance_1,▁
accuracy/tolerance_2,▁
agreement/cohen_kappa,▁
agreement/weighted_f1,▁
correlation/pearson,▁
correlation/pearson_p_value,▁
correlation/spearman,▁
correlation/spearman_p_value,▁
dataset/failed_extractions,▁



=== EXAMPLE PREDICTIONS ===

Example 1:
Predicted Score: 1
Ground Truth Score: 1
Correct: True
Prediction: Score: 1
Rationale: 'Selecting the minimum' (part of 2). Misses sorting, in-place, swapping, and growing sorted part (1, 3, 4).
--------------------------------------------------------------------------------

Example 2:
Predicted Score: 1
Ground Truth Score: 1
Correct: True
Prediction: Score: 1
Rationale: "Makes you unable to walk" describes paralysis (partially point 3). Fails to define it as a viral disease (point 1) or mention nervous system/motor neurons (point 2).
--------------------------------------------------------------------------------

Example 3:
Predicted Score: 2
Ground Truth Score: 2
Correct: True
Prediction: Score: 2
Rationale: Defines as data to teach model (1) and learns from it (2). Misses optional point 3.
--------------------------------------------------------------------------------

Example 4:
Predicted Score: 1
Ground Truth Score: 1
Correct: True
Predi

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
